# 03 · Export & validate

Convert a trained `.keras` model into the formats the serving layer uses and
prove the export is faithful:

1. `.keras` → TFLite (with an optional quantized build)
2. `.keras` → ONNX, matching the `model_cache/` router/expert layout
3. **Parity check** — Keras vs exported graph on the same image
4. Latency benchmark + ready-to-paste `manifest.json` block

Run this every time a model is promoted to `app/models/` or `model_cache/`.

In [ ]:
import sys
from pathlib import Path
import numpy as np

def _notebook_dir() -> Path:
    """Locate notebooks/ whether Jupyter starts at the repo root, notebooks/, or deeper."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "notebooks" / "utils.py").exists():
            return candidate / "notebooks"
        if (candidate / "utils.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the repo (notebooks/utils.py not found)")


NOTEBOOK_DIR = _notebook_dir()
sys.path.insert(0, str(NOTEBOOK_DIR))
import utils

ARTIFACT_DIR = utils.artifact_dir()

# ---- point at the run you want to ship ---------------------------------
MODEL_PATH = ARTIFACT_DIR / "maize_expert_v1.keras"
NAMES_PATH = ARTIFACT_DIR / "maize_expert_v1.class_names.json"
VERSION    = "1.0.0"
KIND       = "expert"          # "expert" -> experts/…, "router" -> router/…
SLUG       = "maize"           # crop name for experts, "router" for the router
onnx_path  = None              # filled in by the ONNX cell if tf2onnx is installed

print("model :", MODEL_PATH)
print("exists:", MODEL_PATH.exists())

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model(MODEL_PATH)
class_names = (
    utils.load_class_names(NAMES_PATH) if NAMES_PATH.exists() else utils.CLASS_NAMES
)
print("input :", model.input_shape)
print("output:", model.output_shape)
print("classes:", class_names)

In [ ]:
tflite_path = utils.export_tflite(model, ARTIFACT_DIR / f"{SLUG}_expert_v1.tflite")
print("float32 tflite:", tflite_path, f"{tflite_path.stat().st_size / 1e6:.2f} MB")

quant_path = utils.export_tflite(
    model, ARTIFACT_DIR / f"{SLUG}_expert_v1_int8.tflite", quantize=True
)
print("quantized tflite:", quant_path, f"{quant_path.stat().st_size / 1e6:.2f} MB")

In [ ]:
# ONNX is the format model_cache/manifest.json references.
# Requires tf2onnx:  pip install tf2onnx onnx
onnx_path = ARTIFACT_DIR / KIND / f"{SLUG}_v1.onnx"
try:
    utils.export_onnx(model, onnx_path)
    print("onnx:", onnx_path, f"{onnx_path.stat().st_size / 1e6:.2f} MB")
except ImportError as exc:
    print("tf2onnx not installed, skipping:", exc)
    onnx_path = None

In [ ]:
# Parity: the exported graphs must agree with Keras on the same input.
rng = np.random.default_rng(0)
dummy = rng.random((1, *utils.IMG_SIZE, 3), dtype="float32")
keras_probs = model.predict(dummy, verbose=0)[0]
keras_idx = int(np.argmax(keras_probs))
print(f"keras : {class_names[keras_idx]} ({keras_probs[keras_idx]:.4f})")

interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
interpreter.set_tensor(input_detail["index"], dummy.astype(input_detail["dtype"]))
interpreter.invoke()
tflite_probs = interpreter.get_tensor(output_detail["index"])[0]
tflite_idx = int(np.argmax(tflite_probs))

max_delta = float(np.max(np.abs(keras_probs - tflite_probs)))
print(f"tflite: {class_names[tflite_idx]} ({tflite_probs[tflite_idx]:.4f})")
print(f"max |Δ|: {max_delta:.6f}")
assert keras_idx == tflite_idx, "Exported model disagrees with Keras — do not ship"
print("✅ parity OK")

In [ ]:
import time

runs = 20
interpreter.set_tensor(input_detail["index"], dummy.astype(input_detail["dtype"]))
interpreter.invoke()  # warm-up

start = time.perf_counter()
for _ in range(runs):
    interpreter.invoke()
elapsed = (time.perf_counter() - start) / runs
print(f"tflite latency: {elapsed * 1000:.1f} ms/image (CPU, single thread)")

In [ ]:
import json

entry = utils.manifest_entry(
    kind=KIND,
    file=f"{KIND}s/{SLUG}_v1.onnx",
    version=VERSION,
    classes=class_names,
)
print(json.dumps(entry, indent=2))

# Promotion checklist:
#   experts/<slug>_v1.onnx  +  model_cache/manifest.json -> version bump
#   or: app/models/<file>   +  app/models/registry.py entry
print("\nShip these files:")
for path in (tflite_path, quant_path, onnx_path):
    if path is not None and Path(path).exists():
        print("  -", Path(path).relative_to(utils.REPO_ROOT))
print("  -", NAMES_PATH.relative_to(utils.REPO_ROOT) if NAMES_PATH.exists() else "(class names missing!)")